In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path
import shutil

BASE = Path("/content/drive/MyDrive/iot/iot device name/network")
PINH = BASE / "Pinh19"

(PINH / "raw_packet_csvs").mkdir(parents=True, exist_ok=True)
(PINH / "labeled_packet_csvs").mkdir(parents=True, exist_ok=True)
(PINH / "feature_csvs").mkdir(parents=True, exist_ok=True)
(PINH / "results").mkdir(parents=True, exist_ok=True)

print("Created:", PINH)

Created: /content/drive/MyDrive/iot/iot device name/network/Pinh19


In [3]:
src_map = BASE / "Okui22" / "device_map.csv"
dst_map = PINH / "device_map.csv"

if src_map.exists():
    shutil.copy2(src_map, dst_map)
    print("Copied:", dst_map)
else:
    print("device_map.csv not found, please check path.")

Copied: /content/drive/MyDrive/iot/iot device name/network/Pinh19/device_map.csv


In [5]:
import pandas as pd

map_path = PINH / "device_map.csv"
device_map = pd.read_csv(map_path)

device_map["ip"] = device_map["ip"].astype(str).str.strip()
device_map["best_mac"] = (
    device_map["best_mac"]
    .astype(str)
    .str.strip()
    .str.lower()
)
device_map["service_name"] = device_map["service_name"].astype(str).str.strip()

display(device_map.head(20))
print(device_map.columns.tolist())

,ip,best_mac,service_name
0,192.168.1.133,dc:56:e7:5b:61:41,service_7
1,192.168.1.146,60:14:b3:b1:91:73,service_2
2,192.168.1.152,00:0c:29:d2:b0:02,service_1
3,192.168.1.17,80:3f:5d:10:17:e1,service_5
4,192.168.1.190,00:0c:29:ee:e0:7a,service_3
5,192.168.1.191,80:3f:5d:10:17:e1,service_5
6,192.168.1.192,60:14:b3:b1:91:73,service_2
7,192.168.1.193,60:14:b3:b1:91:73,service_2
8,192.168.1.194,60:14:b3:b1:91:73,service_2
9,192.168.1.250,d4:dc:cd:b4:26:3e,service_6


['ip', 'best_mac', 'service_name']


In [7]:
import pandas as pd

map_path = PINH / "device_map.csv"
device_map = pd.read_csv(map_path)

device_map["ip"] = device_map["ip"].astype(str).str.strip()
device_map["best_mac"] = (
    device_map["best_mac"]
    .astype(str)
    .str.strip()
    .str.lower()
)
device_map["service_name"] = device_map["service_name"].astype(str).str.strip()

display(device_map.head(20))
print(device_map.columns.tolist())

,ip,best_mac,service_name
0,192.168.1.133,dc:56:e7:5b:61:41,service_7
1,192.168.1.146,60:14:b3:b1:91:73,service_2
2,192.168.1.152,00:0c:29:d2:b0:02,service_1
3,192.168.1.17,80:3f:5d:10:17:e1,service_5
4,192.168.1.190,00:0c:29:ee:e0:7a,service_3
5,192.168.1.191,80:3f:5d:10:17:e1,service_5
6,192.168.1.192,60:14:b3:b1:91:73,service_2
7,192.168.1.193,60:14:b3:b1:91:73,service_2
8,192.168.1.194,60:14:b3:b1:91:73,service_2
9,192.168.1.250,d4:dc:cd:b4:26:3e,service_6


['ip', 'best_mac', 'service_name']


In [8]:
mac_to_device = (
    device_map.dropna(subset=["best_mac", "service_name"])
              .drop_duplicates(subset=["best_mac"])
              .set_index("best_mac")["service_name"]
              .to_dict()
)

ip_to_device = (
    device_map.dropna(subset=["ip", "service_name"])
              .drop_duplicates(subset=["ip"])
              .set_index("ip")["service_name"]
              .to_dict()
)

print("MAC labels:", len(mac_to_device))
print("IP labels :", len(ip_to_device))

MAC labels: 7
IP labels : 12


In [9]:
import subprocess

def extract_packets_from_pcap(pcap_path, out_csv):
    fields = [
        "frame.time_epoch",
        "frame.len",
        "eth.src",
        "eth.dst",
        "ip.src",
        "ip.dst",
        "ip.proto",
        "tcp.srcport",
        "tcp.dstport",
        "udp.srcport",
        "udp.dstport",
    ]
    
    cmd = ["tshark", "-r", str(pcap_path), "-T", "fields"]
    for f in fields:
        cmd += ["-e", f]
    cmd += ["-E", "header=y", "-E", "separator=,", "-E", "quote=d", "-E", "occurrence=f"]

    with open(out_csv, "w", encoding="utf-8") as f:
        subprocess.run(cmd, stdout=f, check=True)

    print("saved:", out_csv)

In [10]:
pcap_path = BASE / "normal_1.pcap"
raw_csv_path = PINH / "raw_packet_csvs" / "normal_1_packets.csv"

extract_packets_from_pcap(pcap_path, raw_csv_path)

raw_df = pd.read_csv(raw_csv_path, low_memory=False)
display(raw_df.head())
print(raw_df.shape)

saved: /content/drive/MyDrive/iot/iot device name/network/Pinh19/raw_packet_csvs/normal_1_packets.csv


,frame.time_epoch,frame.len,eth.src,eth.dst,ip.src,ip.dst,ip.proto,tcp.srcport,tcp.dstport,udp.srcport,udp.dstport
0,1.554220e+09,551,00:00:7f:06:ed:8c,09:00:02:17:04:d2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.554220e+09,552,00:00:7f:06:ec:89,0a:00:02:18:04:d2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1.554220e+09,189,NaN,NaN,192.168.1.152,192.168.1.192,6.0,1880.0,40571.0,NaN,NaN
3,1.554220e+09,189,NaN,NaN,192.168.1.152,192.168.1.190,6.0,1880.0,43539.0,NaN,NaN
4,1.554220e+09,68,NaN,NaN,192.168.1.190,192.168.1.152,6.0,43539.0,1880.0,NaN,NaN


(2000000, 11)


In [11]:
def label_packet_csv(raw_csv_path, labeled_csv_path, mac_to_device, ip_to_device):
    df = pd.read_csv(raw_csv_path, low_memory=False)

    for col in ["eth.src", "eth.dst"]:
        if col in df.columns:
            df[col] = df[col].fillna("").astype(str).str.strip().str.lower()

    for col in ["ip.src", "ip.dst"]:
        if col in df.columns:
            df[col] = df[col].fillna("").astype(str).str.strip()

    df["src_device"] = df["eth.src"].map(mac_to_device)
    df["src_device"] = df["src_device"].fillna(df["ip.src"].map(ip_to_device))

    df["dst_device"] = df["eth.dst"].map(mac_to_device)
    df["dst_device"] = df["dst_device"].fillna(df["ip.dst"].map(ip_to_device))

    # Pinh19-style: transmitted bytes -> use source side as transmitting device
    df["tx_device"] = df["src_device"]

    df.to_csv(labeled_csv_path, index=False)
    print("saved:", labeled_csv_path)
    print(df[["eth.src", "ip.src", "src_device", "eth.dst", "ip.dst", "dst_device", "tx_device"]].head(10))

    return df

In [13]:
labeled_csv_path = PINH / "labeled_packet_csvs" / "normal_1_packets_labeled.csv"

labeled_df = label_packet_csv(
    raw_csv_path,
    labeled_csv_path,
    mac_to_device,
    ip_to_device
)

print(labeled_df["tx_device"].value_counts(dropna=False).head(20))

saved: /content/drive/MyDrive/iot/iot device name/network/Pinh19/labeled_packet_csvs/normal_1_packets_labeled.csv
             eth.src         ip.src src_device            eth.dst  \
0  00:00:7f:06:ed:8c                       NaN  09:00:02:17:04:d2   
1  00:00:7f:06:ec:89                       NaN  0a:00:02:18:04:d2   
2                     192.168.1.152  service_1                      
3                     192.168.1.152  service_1                      
4                     192.168.1.190  service_3                      
5                     192.168.1.152  service_1                      
6                     192.168.1.195        NaN                      
7  00:00:7f:06:eb:86                       NaN  0b:00:02:19:04:d2   
8                     192.168.1.192  service_2                      
9  00:00:7f:06:ea:83                       NaN  0c:00:02:1a:04:d2   

          ip.dst dst_device  tx_device  
0                       NaN        NaN  
1                       NaN        NaN  
2  

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("/content/drive/MyDrive/iot/iot device name/network")
PINH = BASE / "Pinh19"

labeled_csv_path = PINH / "labeled_packet_csvs" / "normal_1_packets_labeled.csv"
feature_csv_path = PINH / "feature_csvs" / "normal_1_pinh_features.csv"

df = pd.read_csv(labeled_csv_path, low_memory=False)

# Retain only packets for which the “sending device” has been identified
df = df.dropna(subset=["tx_device"]).copy()

df["frame.time_epoch"] = pd.to_numeric(df["frame.time_epoch"], errors="coerce")
df["frame.len"] = pd.to_numeric(df["frame.len"], errors="coerce")
df = df.dropna(subset=["frame.time_epoch", "frame.len"]).copy()

# Pinh19: one-second windows
df["ts"] = pd.to_datetime(df["frame.time_epoch"], unit="s", utc=True)
df["window_start"] = df["ts"].dt.floor("1s")

# Pinh19 3 features
feat_df = (
    df.groupby(["tx_device", "window_start"])["frame.len"]
      .agg(
          total_tx_bytes="sum",
          mean_tx_bytes="mean",
          std_tx_bytes=lambda s: s.std(ddof=0)
      )
      .reset_index()
      .rename(columns={"tx_device": "device_name"})
)

feat_df["std_tx_bytes"] = feat_df["std_tx_bytes"].fillna(0.0)

feat_df.to_csv(feature_csv_path, index=False)
print("saved:", feature_csv_path)

display(feat_df.head())
print(feat_df.shape)
print(feat_df["device_name"].value_counts())

saved: /content/drive/MyDrive/iot/iot device name/network/Pinh19/feature_csvs/normal_1_pinh_features.csv


,device_name,window_start,total_tx_bytes,mean_tx_bytes,std_tx_bytes
0,service_1,2019-04-02 15:52:04+00:00,5073,563.666667,340.233580
1,service_1,2019-04-02 15:52:05+00:00,17156,612.714286,251.291359
2,service_1,2019-04-02 15:52:06+00:00,21436,612.457143,411.658898
3,service_1,2019-04-02 15:52:07+00:00,26608,760.228571,691.398834
4,service_1,2019-04-02 15:52:08+00:00,6496,499.692308,284.248370


(29721, 5)
device_name
service_3    9803
service_2    9800
service_1    9797
service_6     191
service_7     107
service_4      23
Name: count, dtype: int64


In [15]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/iot/iot device name/network")
PINH = BASE / "Pinh19"

for sub in ["raw_packet_csvs", "labeled_packet_csvs", "feature_csvs", "results"]:
    (PINH / sub).mkdir(parents=True, exist_ok=True)

print("ready:", PINH)

ready: /content/drive/MyDrive/iot/iot device name/network/Pinh19


In [ ]:
import pandas as pd
import numpy as np
import subprocess
from pathlib import Path

# ---------- 读取 device map ----------
def load_device_map(map_path):
    device_map = pd.read_csv(map_path)

    device_map["ip"] = device_map["ip"].astype(str).str.strip()
    device_map["best_mac"] = device_map["best_mac"].astype(str).str.strip().str.lower()
    device_map["service_name"] = device_map["service_name"].astype(str).str.strip()

    mac_to_device = (
        device_map.dropna(subset=["best_mac", "service_name"])
                  .drop_duplicates(subset=["best_mac"])
                  .set_index("best_mac")["service_name"]
                  .to_dict()
    )

    ip_to_device = (
        device_map.dropna(subset=["ip", "service_name"])
                  .drop_duplicates(subset=["ip"])
                  .set_index("ip")["service_name"]
                  .to_dict()
    )

    return device_map, mac_to_device, ip_to_device


# ---------- Extract packets from a PCAP file ----------
def extract_packets_from_pcap(pcap_path, out_csv):
    fields = [
        "frame.time_epoch",
        "frame.len",
        "eth.src",
        "eth.dst",
        "ip.src",
        "ip.dst",
        "ip.proto",
        "tcp.srcport",
        "tcp.dstport",
        "udp.srcport",
        "udp.dstport",
    ]

    cmd = ["tshark", "-r", str(pcap_path), "-T", "fields"]
    for f in fields:
        cmd += ["-e", f]
    cmd += ["-E", "header=y", "-E", "separator=,", "-E", "quote=d", "-E", "occurrence=f"]

    with open(out_csv, "w", encoding="utf-8") as f:
        subprocess.run(cmd, stdout=f, check=True)

    return out_csv


# ---------- labeling ----------
def label_packet_csv(raw_csv_path, labeled_csv_path, mac_to_device, ip_to_device):
    df = pd.read_csv(raw_csv_path, low_memory=False)

    for col in ["eth.src", "eth.dst"]:
        if col in df.columns:
            df[col] = df[col].fillna("").astype(str).str.strip().str.lower()

    for col in ["ip.src", "ip.dst"]:
        if col in df.columns:
            df[col] = df[col].fillna("").astype(str).str.strip()

    df["src_device_by_mac"] = df["eth.src"].map(mac_to_device)
    df["src_device_by_ip"] = df["ip.src"].map(ip_to_device)
    df["tx_device"] = df["src_device_by_mac"].fillna(df["src_device_by_ip"])

    df["dst_device_by_mac"] = df["eth.dst"].map(mac_to_device)
    df["dst_device_by_ip"] = df["ip.dst"].map(ip_to_device)
    df["rx_device"] = df["dst_device_by_mac"].fillna(df["dst_device_by_ip"])

    df.to_csv(labeled_csv_path, index=False)
    return df


# ---------- Feature extraction ----------
def build_pinh_features(labeled_csv_path, feature_csv_path):
    df = pd.read_csv(labeled_csv_path, low_memory=False)

    df = df.dropna(subset=["tx_device"]).copy()

    df["frame.time_epoch"] = pd.to_numeric(df["frame.time_epoch"], errors="coerce")
    df["frame.len"] = pd.to_numeric(df["frame.len"], errors="coerce")
    df = df.dropna(subset=["frame.time_epoch", "frame.len"]).copy()

    df["ts"] = pd.to_datetime(df["frame.time_epoch"], unit="s", utc=True)
    df["window_start"] = df["ts"].dt.floor("1s")

    feat_df = (
        df.groupby(["tx_device", "window_start"])["frame.len"]
          .agg(
              total_tx_bytes="sum",
              mean_tx_bytes="mean",
              std_tx_bytes=lambda s: s.std(ddof=0)
          )
          .reset_index()
          .rename(columns={"tx_device": "device_name"})
    )

    feat_df["std_tx_bytes"] = feat_df["std_tx_bytes"].fillna(0.0)
    feat_df["source_file"] = Path(labeled_csv_path).stem

    feat_df.to_csv(feature_csv_path, index=False)
    return feat_df

In [17]:
from pathlib import Path
import pandas as pd

map_path = PINH / "device_map.csv"
device_map, mac_to_device, ip_to_device = load_device_map(map_path)

pcap_files = [BASE / f"normal_{i}.pcap" for i in range(1, 14)]
pcap_files = [p for p in pcap_files if p.exists()]

print("num pcaps found:", len(pcap_files))
for p in pcap_files:
    print("-", p.name)

num pcaps found: 13
- normal_1.pcap
- normal_2.pcap
- normal_3.pcap
- normal_4.pcap
- normal_5.pcap
- normal_6.pcap
- normal_7.pcap
- normal_8.pcap
- normal_9.pcap
- normal_10.pcap
- normal_11.pcap
- normal_12.pcap
- normal_13.pcap


In [18]:
summary_rows = []

for pcap_path in pcap_files:
    stem = pcap_path.stem

    raw_csv_path = PINH / "raw_packet_csvs" / f"{stem}_packets.csv"
    labeled_csv_path = PINH / "labeled_packet_csvs" / f"{stem}_packets_labeled.csv"
    feature_csv_path = PINH / "feature_csvs" / f"{stem}_pinh_features.csv"

    print(f"\n=== Processing {stem} ===")

    # 1) pcap -> raw csv
    if not raw_csv_path.exists():
        extract_packets_from_pcap(pcap_path, raw_csv_path)
        print("  raw saved")
    else:
        print("  raw exists")

    # 2) raw -> labeled
    labeled_df = label_packet_csv(raw_csv_path, labeled_csv_path, mac_to_device, ip_to_device)
    tx_nonnull = labeled_df["tx_device"].notna().sum()
    print("  labeled saved")
    print("  tx_device non-null:", tx_nonnull)

    # 3) labeled -> pinh features
    feat_df = build_pinh_features(labeled_csv_path, feature_csv_path)
    print("  feature saved")
    print("  feature rows:", len(feat_df))
    print("  classes:", feat_df["device_name"].nunique())

    vc = feat_df["device_name"].value_counts().to_dict()

    summary_rows.append({
        "pcap": stem,
        "raw_rows": len(labeled_df),
        "tx_device_nonnull_rows": tx_nonnull,
        "feature_rows": len(feat_df),
        "n_classes": feat_df["device_name"].nunique(),
        **{f"class_{k}": v for k, v in vc.items()}
    })

print("\nAll done.")


=== Processing normal_1 ===
  raw exists
  labeled saved
  tx_device non-null: 347500
  feature saved
  feature rows: 29721
  classes: 6

=== Processing normal_2 ===
  raw saved
  labeled saved
  tx_device non-null: 206439
  feature saved
  feature rows: 33090
  classes: 6

=== Processing normal_3 ===
  raw saved
  labeled saved
  tx_device non-null: 197931
  feature saved
  feature rows: 33463
  classes: 5

=== Processing normal_4 ===
  raw saved
  labeled saved
  tx_device non-null: 215893
  feature saved
  feature rows: 36799
  classes: 7

=== Processing normal_5 ===
  raw saved
  labeled saved
  tx_device non-null: 207349
  feature saved
  feature rows: 39063
  classes: 7

=== Processing normal_6 ===
  raw saved
  labeled saved
  tx_device non-null: 204743
  feature saved
  feature rows: 39253
  classes: 7

=== Processing normal_7 ===
  raw saved
  labeled saved
  tx_device non-null: 145919
  feature saved
  feature rows: 26970
  classes: 5

=== Processing normal_8 ===
  raw saved

In [19]:
feature_files = sorted((PINH / "feature_csvs").glob("*_pinh_features.csv"))
print("feature files:", len(feature_files))

all_feat = pd.concat([pd.read_csv(f) for f in feature_files], ignore_index=True)
all_feat["window_start"] = pd.to_datetime(all_feat["window_start"], errors="coerce", utc=True)
all_feat = all_feat.dropna(subset=["window_start", "device_name"]).copy()

all_feat_path = PINH / "results" / "all_pinh_features.csv"
all_feat.to_csv(all_feat_path, index=False)

print("saved:", all_feat_path)
display(all_feat.head())
print(all_feat.shape)
print(all_feat["device_name"].value_counts())

feature files: 13
saved: /content/drive/MyDrive/iot/iot device name/network/Pinh19/results/all_pinh_features.csv


,device_name,window_start,total_tx_bytes,mean_tx_bytes,std_tx_bytes,source_file
0,service_1,2019-04-03 20:38:05+00:00,13778,1252.545455,1473.409612,normal_10_packets_labeled
1,service_1,2019-04-03 20:38:06+00:00,5534,691.750000,913.910383,normal_10_packets_labeled
2,service_1,2019-04-03 20:38:07+00:00,2655,379.285714,285.521435,normal_10_packets_labeled
3,service_1,2019-04-03 20:38:08+00:00,1585,264.166667,299.545443,normal_10_packets_labeled
4,service_1,2019-04-03 20:38:09+00:00,5826,728.250000,862.503442,normal_10_packets_labeled


(386223, 6)
device_name
service_3    143726
service_1    143632
service_2     67214
service_4     26567
service_7      1789
service_6      1771
service_5      1524
Name: count, dtype: int64


In [20]:
summary_df = pd.DataFrame(summary_rows)
summary_path = PINH / "results" / "feature_build_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("saved:", summary_path)
display(summary_df)

saved: /content/drive/MyDrive/iot/iot device name/network/Pinh19/results/feature_build_summary.csv


,pcap,raw_rows,tx_device_nonnull_rows,feature_rows,n_classes,class_service_3,class_service_2,class_service_1,class_service_6,class_service_7,class_service_4,class_service_5
0,normal_1,2000000,347500,29721,6,9803,9800.0,9797,191.0,107.0,23.0,NaN
1,normal_2,2000000,206439,33090,6,10877,10875.0,10877,53.0,316.0,92.0,NaN
2,normal_3,2000000,197931,33463,5,11146,11057.0,11148,42.0,70.0,NaN,NaN
3,normal_4,2000000,215893,36799,7,10967,10663.0,10967,206.0,171.0,3482.0,343.0
4,normal_5,2000000,207349,39063,7,10925,10447.0,10925,93.0,226.0,6066.0,381.0
5,normal_6,2000000,204743,39253,7,10897,9932.0,10803,187.0,251.0,6714.0,469.0
6,normal_7,2000000,145919,26970,5,11302,NaN,11302,587.0,228.0,3551.0,NaN
7,normal_8,2000000,136778,22754,5,11229,NaN,11230,196.0,85.0,14.0,NaN
8,normal_9,2000000,135699,23085,5,11472,NaN,11473,53.0,63.0,24.0,NaN
9,normal_10,2000000,138148,23109,4,11516,NaN,11516,45.0,NaN,32.0,NaN


In [46]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import label_binarize
from sklearn.metrics import average_precision_score, classification_report

# 读数据
df = pd.read_csv("/content/drive/MyDrive/iot/iot device name/network/Pinh19/results/all_pinh_features.csv")

# 清洗
df["window_start"] = pd.to_datetime(df["window_start"], errors="coerce", utc=True)
for c in ["total_tx_bytes", "mean_tx_bytes", "std_tx_bytes"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=[
    "window_start", "device_name", "source_file",
    "total_tx_bytes", "mean_tx_bytes", "std_tx_bytes"
]).copy()

FEATURES = ["total_tx_bytes", "mean_tx_bytes", "std_tx_bytes"]
X = df[FEATURES]
y = df["device_name"]
groups = df["source_file"]

def multiclass_aucpr(y_true, proba, classes):
    Y = label_binarize(y_true, classes=classes)
    if len(classes) == 2 and Y.shape[1] == 1:
        Y = np.hstack([1 - Y, Y])

    macro = average_precision_score(Y, proba, average="macro")
    weighted = average_precision_score(Y, proba, average="weighted")
    micro = average_precision_score(Y, proba, average="micro")
    return macro, weighted, micro

gkf = GroupKFold(n_splits=7)

target_fold =4

for fold, (tr_idx, te_idx) in enumerate(gkf.split(X, y, groups=groups), start=1):
    if fold != target_fold:
        continue

    X_train, X_test = X.iloc[tr_idx], X.iloc[te_idx]
    y_train, y_test = y.iloc[tr_idx], y.iloc[te_idx]

    train_files = sorted(df.iloc[tr_idx]["source_file"].unique())
    test_files = sorted(df.iloc[te_idx]["source_file"].unique())

    print("Fold:", fold)
    print("train files:", train_files)
    print("test files :", test_files)
    print()
    print("train shape:", X_train.shape)
    print("test shape :", X_test.shape)
    print()
    print("train classes:")
    print(y_train.value_counts())
    print()
    print("test classes:")
    print(y_test.value_counts())

    clf = RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    )
    clf.fit(X_train, y_train)

    pred = clf.predict(X_test)
    proba = clf.predict_proba(X_test)
    classes = clf.classes_

    macro_aucpr, weighted_aucpr, micro_aucpr = multiclass_aucpr(y_test, proba, classes)

    print()
    print("Macro AUCPR   :", macro_aucpr)
    print("Weighted AUCPR:", weighted_aucpr)
    print("Micro AUCPR   :", micro_aucpr)
    print()
    print(classification_report(y_test, pred, zero_division=0))

    break

Fold: 4
train files: ['normal_10_packets_labeled', 'normal_11_packets_labeled', 'normal_12_packets_labeled', 'normal_13_packets_labeled', 'normal_1_packets_labeled', 'normal_2_packets_labeled', 'normal_4_packets_labeled', 'normal_5_packets_labeled', 'normal_6_packets_labeled', 'normal_7_packets_labeled', 'normal_8_packets_labeled']
test files : ['normal_3_packets_labeled', 'normal_9_packets_labeled']

train shape: (329675, 3)
test shape : (56548, 3)

train classes:
device_name
service_3    121108
service_1    121011
service_2     56157
service_4     26543
service_6      1676
service_7      1656
service_5      1524
Name: count, dtype: int64

test classes:
device_name
service_1    22621
service_3    22618
service_2    11057
service_7      133
service_6       95
service_4       24
Name: count, dtype: int64


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(



Macro AUCPR   : 0.8400004806248498
Weighted AUCPR: 0.99990128514576
Micro AUCPR   : 0.9999829021789997

              precision    recall  f1-score   support

   service_1       1.00      1.00      1.00     22621
   service_2       1.00      1.00      1.00     11057
   service_3       1.00      1.00      1.00     22618
   service_4       0.77      0.83      0.80        24
   service_5       0.00      0.00      0.00         0
   service_6       0.88      0.96      0.92        95
   service_7       0.98      0.90      0.94       133

    accuracy                           1.00     56548
   macro avg       0.80      0.81      0.81     56548
weighted avg       1.00      1.00      1.00     56548



In [ ]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import label_binarize
from sklearn.metrics import average_precision_score, classification_report


path = "/content/drive/MyDrive/iot/iot device name/network/Pinh19/results/all_pinh_features.csv"
df = pd.read_csv(path)


df["window_start"] = pd.to_datetime(df["window_start"], errors="coerce", utc=True)
for c in ["total_tx_bytes", "mean_tx_bytes", "std_tx_bytes"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=[
    "window_start", "device_name", "source_file",
    "total_tx_bytes", "mean_tx_bytes", "std_tx_bytes"
]).copy()



hours = df["window_start"].dt.hour
df["mode"] = np.where((hours >= 2) & (hours < 6), "active", "idle")

print(df["mode"].value_counts())
print(df[["window_start", "device_name", "source_file", "mode"]].head())

FEATURES = ["total_tx_bytes", "mean_tx_bytes", "std_tx_bytes"]
X = df[FEATURES]
y = df["device_name"]
groups = df["source_file"]

def multiclass_aucpr(y_true, proba, classes):
    Y = label_binarize(y_true, classes=classes)
    if len(classes) == 2 and Y.shape[1] == 1:
        Y = np.hstack([1 - Y, Y])

    macro = average_precision_score(Y, proba, average="macro")
    weighted = average_precision_score(Y, proba, average="weighted")
    micro = average_precision_score(Y, proba, average="micro")
    return macro, weighted, micro

def eval_on_subset(clf, test_df, scenario_name):
    """
    scenario_name: idle / active / mix
    """
    if scenario_name == "mix":
        sub = test_df.copy()
    else:
        sub = test_df[test_df["mode"] == scenario_name].copy()

    if len(sub) == 0:
        return None

    X_test = sub[FEATURES]
    y_test = sub["device_name"]

  
    seen_classes = clf.classes_
    mask = y_test.isin(seen_classes)
    sub = sub[mask].copy()

    if len(sub) == 0:
        return None

    X_test = sub[FEATURES]
    y_test = sub["device_name"]

    
    if y_test.nunique() < 2:
        return None

    pred = clf.predict(X_test)
    proba = clf.predict_proba(X_test)
    classes = clf.classes_

    macro_aucpr, weighted_aucpr, micro_aucpr = multiclass_aucpr(y_test, proba, classes)

    return {
        "scenario": scenario_name,
        "n_test": len(sub),
        "n_classes_test": y_test.nunique(),
        "macro_aucpr": macro_aucpr,
        "weighted_aucpr": weighted_aucpr,
        "micro_aucpr": micro_aucpr,
        "report": classification_report(y_test, pred, zero_division=0)
    }

# ========= GroupKFold: Train=Mix, Test=Idle/Active/Mix =========
gkf = GroupKFold(n_splits=10)

rows = []
reports = {}

for fold, (tr_idx, te_idx) in enumerate(gkf.split(X, y, groups=groups), start=1):
    train_df = df.iloc[tr_idx].copy()
    test_df = df.iloc[te_idx].copy()

    X_train = train_df[FEATURES]
    y_train = train_df["device_name"]

    clf = RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    )
    clf.fit(X_train, y_train)

    train_files = sorted(train_df["source_file"].unique())
    test_files = sorted(test_df["source_file"].unique())

    for scenario in ["idle", "active", "mix"]:
        res = eval_on_subset(clf, test_df, scenario)
        if res is None:
            continue

        rows.append({
            "fold": fold,
            "scenario": scenario,
            "n_train": len(train_df),
            "n_test": res["n_test"],
            "n_classes_train": y_train.nunique(),
            "n_classes_test": res["n_classes_test"],
            "train_files": len(train_files),
            "test_files": len(test_files),
            "train_file_names": ", ".join(train_files),
            "test_file_names": ", ".join(test_files),
            "macro_aucpr": res["macro_aucpr"],
            "weighted_aucpr": res["weighted_aucpr"],
            "micro_aucpr": res["micro_aucpr"],
        })

        reports[(fold, scenario)] = res["report"]

mode_cv_df = pd.DataFrame(rows)
display(mode_cv_df)

# ========= average =========
summary = (
    mode_cv_df.groupby("scenario", as_index=False)
              .agg(
                  folds=("fold", "count"),
                  macro_aucpr_mean=("macro_aucpr", "mean"),
                  macro_aucpr_std=("macro_aucpr", lambda s: s.std(ddof=0)),
                  weighted_aucpr_mean=("weighted_aucpr", "mean"),
                  weighted_aucpr_std=("weighted_aucpr", lambda s: s.std(ddof=0)),
                  micro_aucpr_mean=("micro_aucpr", "mean"),
                  micro_aucpr_std=("micro_aucpr", lambda s: s.std(ddof=0)),
              )
)

display(summary)


out1 = "/content/drive/MyDrive/iot/iot device name/network/Pinh19/results/pinh19_mode_operation_fold_results.csv"
out2 = "/content/drive/MyDrive/iot/iot device name/network/Pinh19/results/pinh19_mode_operation_summary.csv"

mode_cv_df.to_csv(out1, index=False)
summary.to_csv(out2, index=False)

print("saved:", out1)
print("saved:", out2)

mode
idle      304186
active     82037
Name: count, dtype: int64
               window_start device_name                source_file  mode
0 2019-04-03 20:38:05+00:00   service_1  normal_10_packets_labeled  idle
1 2019-04-03 20:38:06+00:00   service_1  normal_10_packets_labeled  idle
2 2019-04-03 20:38:07+00:00   service_1  normal_10_packets_labeled  idle
3 2019-04-03 20:38:08+00:00   service_1  normal_10_packets_labeled  idle
4 2019-04-03 20:38:09+00:00   service_1  normal_10_packets_labeled  idle


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive c

,fold,scenario,n_train,n_test,n_classes_train,n_classes_test,train_files,test_files,train_file_names,test_file_names,macro_aucpr,weighted_aucpr,micro_aucpr
0,1,idle,346970,39253,7,7,12,1,"normal_10_packets_labeled, normal_11_packets_l...",normal_6_packets_labeled,0.973407,0.998424,0.999235
1,1,mix,346970,39253,7,7,12,1,"normal_10_packets_labeled, normal_11_packets_l...",normal_6_packets_labeled,0.973407,0.998424,0.999235
2,2,idle,347160,9795,7,7,12,1,"normal_10_packets_labeled, normal_11_packets_l...",normal_5_packets_labeled,0.968132,0.998124,0.998945
3,2,active,347160,29268,7,7,12,1,"normal_10_packets_labeled, normal_11_packets_l...",normal_5_packets_labeled,0.995326,0.999586,0.999869
4,2,mix,347160,39063,7,7,12,1,"normal_10_packets_labeled, normal_11_packets_l...",normal_5_packets_labeled,0.988718,0.999182,0.999638
5,3,idle,349424,14353,7,7,12,1,"normal_10_packets_labeled, normal_11_packets_l...",normal_4_packets_labeled,0.953317,0.997147,0.998664
6,3,active,349424,22446,7,7,12,1,"normal_10_packets_labeled, normal_11_packets_l...",normal_4_packets_labeled,0.968368,0.998813,0.999910
7,3,mix,349424,36799,7,7,12,1,"normal_10_packets_labeled, normal_11_packets_l...",normal_4_packets_labeled,0.948767,0.997745,0.999427
8,4,idle,352760,33463,7,5,12,1,"normal_10_packets_labeled, normal_11_packets_l...",normal_3_packets_labeled,0.706686,0.999925,0.999998
9,4,mix,352760,33463,7,5,12,1,"normal_10_packets_labeled, normal_11_packets_l...",normal_3_packets_labeled,0.706686,0.999925,0.999998


,scenario,folds,macro_aucpr_mean,macro_aucpr_std,weighted_aucpr_mean,weighted_aucpr_std,micro_aucpr_mean,micro_aucpr_std
0,active,3,0.844495,0.194557,0.999404,0.000428,0.999863,0.000041
1,idle,10,0.822126,0.130343,0.998697,0.001072,0.999203,0.000765
2,mix,10,0.834509,0.122464,0.998862,0.000981,0.999337,0.000733


saved: /content/drive/MyDrive/iot/iot device name/network/Pinh19/results/pinh19_mode_operation_fold_results.csv
saved: /content/drive/MyDrive/iot/iot device name/network/Pinh19/results/pinh19_mode_operation_summary.csv
